In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession\
        .builder\
        .appName("Cours de Spark")\
        .master("local[*]")\
        .getOrCreate()
sc = spark.sparkContext

df = spark.read.option("header", True)\
               .option("inferSchema", True)\
               .option("escape", "\"")\
               .csv("gps_app.csv")

df_20 = df.withColumn("Rating/20", 4 * col("Rating"))

df_renamed = df_20.withColumnRenamed("Reviews", "Nbr of Reviews")

df_used = df_renamed.withColumn("used", when(col("Nbr of Reviews") >= 10000, True).\
                                        otherwise(False))

df_unique = df_used.dropDuplicates(['App'])

df_clean = df_unique.drop("Rating")\
                    .dropna(subset=["Type", "Content Rating", "Android Ver"])

df_clean = df_clean.filter(~isnan("Type") & ~isnan("Android Ver"))

rating_average_df = df_clean.filter(~isnan("Rating/20")).select(avg("Rating/20"))
rating_average = rating_average_df.head()["avg(Rating/20)"]
df_nomiss = df_clean.fillna({"Rating/20": rating_average})

grouped_ver = df_nomiss.groupBy("Current Ver").count().sort("count", ascending=False)
mode_ver = grouped_ver.head()["Current Ver"]
df_final = df_nomiss.withColumn("Current Ver", when(isnan("Current Ver") | isnull("Current Ver"), mode_ver)\
                                              .otherwise("Current Ver"))

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [2]:
df_final.write.parquet("gps_app.parquet")

In [3]:
df_par = spark.read.parquet("gps_app.parquet")


In [4]:
from pyspark.sql import SparkSession

spark = SparkSession\
        .builder\
        .appName("Cours de Spark")\
        .master("local[*]")\
        .getOrCreate()
sc = spark.sparkContext

aList = sc.parallelize([["Romain Gary","La promesse de l'Aube"],
                        ["Hervé Bazin","Vipère au poing"],
                        ["Victor Hugo","Les Misérables"]])

aList.toDF().write.format("avro").save("books.avro") 

aDF = spark.read.format("avro").load("books.avro")

AnalysisException: Failed to find data source: avro. Avro is built-in but external data source module since Spark 2.4. Please deploy the application as per the deployment section of Apache Avro Data Source Guide.